# exp_014 v2：长周期 RankGLU 残差修正

本 Notebook 使用 exp_009 的三段 OOF anchor，训练 20 期时序 RankGLU 作为标准化残差修正器。训练、推理统一使用 `rank(zscore(anchor) + alpha * zscore(residual))`，避免把残差误当成独立专家。

默认 `RUN_MODE="full"`；设置 `DSCR_EXP014_V2_MODE=smoke` 只执行合成安全检查，设置为 `preflight` 执行真实数据契约检查。v1 与正式提交文件均保持只读。

In [8]:
from __future__ import annotations

import hashlib
import csv
import gc
import copy
import tempfile
import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata

try:
    from IPython.display import display
except Exception:
    display = print

# Keep the logical working-directory spelling. On Windows, resolve()
# would turn the ASCII junction `youanbei` back into its Chinese target,
# which can break native libraries such as LightGBM.
PROJECT_ROOT = Path.cwd().absolute()
if not (PROJECT_ROOT / "03_cache" / "processed_data_v1").exists():
    for candidate in [Path(__file__).resolve().parents[2] if "__file__" in globals() else None, Path(r"D:\google_dl\book\youanbei")]:
        if candidate is not None and (candidate / "03_cache" / "processed_data_v1").exists():
            PROJECT_ROOT = candidate.absolute()
            break

CACHE_DIR = PROJECT_ROOT / "03_cache" / "processed_data_v1"
EXP009_DIR = PROJECT_ROOT / "04_results" / "exp_009_anchor_recent_blend_cv"
EXP009_CACHE = EXP009_DIR / "runtime_cache"
FINAL_SUBMISSION = PROJECT_ROOT / "04_results" / "final_submission" / "prediction.npy"
RUN_DIR = PROJECT_ROOT / "04_results" / "exp_014_anchor_residual_rankglu_v2"

# Full training is now the default requested entry point. Set these
# back to smoke/False if you only want validation.
RUN_MODE = os.environ.get("DSCR_EXP014_V2_MODE", "full").strip().lower()
TRAIN_ENABLED = RUN_MODE == "full"
ANCHOR_MODE = "reuse_then_rebuild_inference"
WINDOW = 20
CURRENT_FEATURES = 328
SEQUENCE_FEATURES = 40
SEEDS = (42, 2026, 3407)
RESIDUAL_ALPHAS = (0.00, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30, 0.50)
DIAGNOSTIC_NEGATIVE_ALPHAS = (-0.20, -0.10, -0.05)
EXP009_WEIGHT = 0.25
MAX_EPOCHS = 25
MIN_EPOCHS = 5
EARLY_STOP_PATIENCE = 4
LEARNING_RATE = 5e-4
MIN_LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
EARLY_STOP_ALPHA = 0.10
FULL_BATCH_SIZE = 8192
FULL_SEEDS = SEEDS
RESUME = True
CHECKPOINT_SCHEMA_VERSION = 4
TRAIN_TIME_ORIGIN = 486
OOF_INTERVALS = {
    "fold_1": (2189, 2432),
    "fold_2": (2432, 2675),
    "fold_3": (2675, 2918),
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
    PRIMARY_PRECISION = "bf16"
    AMP_DTYPE = torch.bfloat16
elif DEVICE.type == "cuda":
    PRIMARY_PRECISION = "fp16"
    AMP_DTYPE = torch.float16
else:
    PRIMARY_PRECISION = "fp32"
    AMP_DTYPE = None
USE_GRAD_SCALER = PRIMARY_PRECISION == "fp16"
STABILITY_POLICY = "finite_checks_amp_then_fp32_retry_v1"

torch.set_float32_matmul_precision("high")
print({"project_root": str(PROJECT_ROOT), "device": str(DEVICE), "run_mode": RUN_MODE, "train_enabled": TRAIN_ENABLED})

{'project_root': 'D:\\google_dl\\book\\youanbei', 'device': 'cuda', 'run_mode': 'full', 'train_enabled': True}


In [9]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(False)


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def finite_array(arr: np.ndarray, name: str) -> None:
    if not np.isfinite(np.asarray(arr)).all():
        raise ValueError(f"{name} contains non-finite values")


def group_rank(values: np.ndarray, group_sizes: Sequence[int]) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    groups = np.asarray(group_sizes, dtype=np.int64).reshape(-1)
    if values.size != int(groups.sum()):
        raise ValueError(f"rank length mismatch: {values.size} != {int(groups.sum())}")
    out = np.empty_like(values, dtype=np.float32)
    offset = 0
    for size in groups:
        size = int(size)
        if size <= 0:
            raise ValueError("empty group")
        block = values[offset : offset + size]
        out[offset : offset + size] = rankdata(block, method="average").astype(np.float32) / float(size)
        offset += size
    return out


def group_zscore(values: torch.Tensor, group_sizes: Sequence[int], eps: float = 1e-5) -> torch.Tensor:
    values = values.float().reshape(-1)
    parts = []
    offset = 0
    for size in group_sizes:
        size = int(size)
        block = values[offset : offset + size]
        mean = block.mean()
        std = block.std(unbiased=False).clamp_min(eps)
        parts.append((block - mean) / std)
        offset += size
    if offset != values.numel():
        raise ValueError("tensor/group size mismatch")
    return torch.cat(parts)


def assert_prediction_grid(pred: np.ndarray, groups: np.ndarray, name: str) -> dict[str, Any]:
    pred = np.asarray(pred)
    if pred.ndim != 2 or pred.shape[1] != 5282:
        raise ValueError(f"{name} must have shape (time, 5282), got {pred.shape}")
    if pred.shape[0] != len(groups):
        raise ValueError(f"{name} time mismatch: {pred.shape[0]} != {len(groups)}")
    finite_array(pred, name)
    unique_ratio = float(np.unique(pred).size / max(pred.size, 1))
    return {"shape": list(pred.shape), "dtype": str(pred.dtype), "unique_ratio": unique_ratio}


def json_dump(path: Path, payload: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    os.replace(temporary, path)


def atomic_torch_save(payload: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def atomic_npz_save(path: Path, **arrays: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)

In [10]:
@dataclass
class CacheContract:
    manifest: dict[str, Any]
    arrays: dict[str, dict[str, Any]]
    splits: dict[str, dict[str, Any]]


def _array_info(path: Path, expected_ndim: int | None = None, expected_last: int | None = None) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(path)
    arr = np.load(path, mmap_mode="r", allow_pickle=False)
    if expected_ndim is not None and arr.ndim != expected_ndim:
        raise ValueError(f"{path.name}: ndim {arr.ndim} != {expected_ndim}")
    if expected_last is not None and arr.shape[-1] != expected_last:
        raise ValueError(f"{path.name}: last dim {arr.shape[-1]} != {expected_last}")
    if arr.dtype.kind not in "fiub":
        raise TypeError(f"{path.name}: unsupported dtype {arr.dtype}")
    return {"shape": list(arr.shape), "dtype": str(arr.dtype), "bytes": int(path.stat().st_size)}


def validate_processed_cache() -> CacheContract:
    ready = CACHE_DIR / "READY"
    manifest_path = CACHE_DIR / "manifest.json"
    if not ready.exists() or not manifest_path.exists():
        raise FileNotFoundError("processed_data_v1 READY/manifest.json is missing")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    arrays = {
        "tree_train": _array_info(CACHE_DIR / "tree" / "train_X.npy", 2, 419),
        "tree_valid": _array_info(CACHE_DIR / "tree" / "valid_X.npy", 2, 419),
        "tree_test": _array_info(CACHE_DIR / "tree" / "test_X.npy", 2, 419),
        "linear_train": _array_info(CACHE_DIR / "linear" / "train_X.npy", 2),
        "linear_valid": _array_info(CACHE_DIR / "linear" / "valid_X.npy", 2),
        "linear_test": _array_info(CACHE_DIR / "linear" / "test_X.npy", 2),
        "sequence": _array_info(CACHE_DIR / "sequence" / "X.npy", 3, SEQUENCE_FEATURES),
        "sequence_mask": _array_info(CACHE_DIR / "sequence" / "mask_x.npy", 2),
    }
    sequence = np.load(CACHE_DIR / "sequence" / "X.npy", mmap_mode="r", allow_pickle=False)
    mask = np.load(CACHE_DIR / "sequence" / "mask_x.npy", mmap_mode="r", allow_pickle=False)
    if sequence.shape[:2] != mask.shape:
        raise ValueError(f"sequence/mask shape mismatch: {sequence.shape} vs {mask.shape}")
    if sequence.shape[0] != 3603 or sequence.shape[1] != 5282:
        raise ValueError(f"unexpected sequence shape: {sequence.shape}")

    splits: dict[str, dict[str, Any]] = {}
    for split in ("train", "valid", "test"):
        group_path = CACHE_DIR / "common" / f"{split}_group_sizes.npy"
        time_path = CACHE_DIR / "common" / f"{split}_time.npy"
        stock_path = CACHE_DIR / "common" / f"{split}_stock.npy"
        groups = np.load(group_path, allow_pickle=False)
        times = np.load(time_path, allow_pickle=False)
        stocks = np.load(stock_path, allow_pickle=False)
        if groups.ndim != 1 or groups.size == 0 or np.any(groups <= 0):
            raise ValueError(f"invalid {split} group_sizes")
        expected_rows = int(groups.sum())
        if len(times) != expected_rows or len(stocks) != expected_rows:
            raise ValueError(f"{split} common arrays do not match group sum")
        if not np.isfinite(times).all() or not np.isfinite(stocks).all():
            raise ValueError(f"{split} time/stock contains non-finite values")
        time_min, time_max = int(times.min()), int(times.max())
        if time_max - time_min + 1 != groups.size:
            raise ValueError(f"{split} time range is not contiguous")
        offset = 0
        for local_time, raw_size in enumerate(groups):
            size = int(raw_size)
            block_times = times[offset : offset + size]
            block_stocks = stocks[offset : offset + size]
            expected_time = time_min + local_time
            if not np.all(block_times == expected_time):
                raise ValueError(f"{split} row order/time groups disagree at time {expected_time}")
            if np.any(np.diff(block_stocks) <= 0):
                raise ValueError(f"{split} stocks are not strictly ordered/unique at time {expected_time}")
            if int(block_stocks[0]) < 0 or int(block_stocks[-1]) >= 5282:
                raise ValueError(f"{split} stock index out of range at time {expected_time}")
            offset += size
        if offset != expected_rows:
            raise ValueError(f"{split} grouped row cursor mismatch")
        if split in ("train", "valid"):
            y = np.load(CACHE_DIR / "common" / f"{split}_y.npy", mmap_mode="r", allow_pickle=False)
            if len(y) != expected_rows:
                raise ValueError(f"{split} y length mismatch")
            finite_array(y, f"{split}_y")
        if np.any(np.diff(times) < 0):
            raise ValueError(f"{split}_time is not nondecreasing")
        splits[split] = {
            "groups": groups.astype(np.int64),
            "rows": expected_rows,
            "n_times": int(groups.size),
            "time_min": int(times.min()),
            "time_max": int(times.max()),
            "stock_min": int(stocks.min()),
            "stock_max": int(stocks.max()),
        }

    if arrays["tree_train"]["shape"][0] != splits["train"]["rows"]:
        raise ValueError("tree train rows do not match train groups")
    if arrays["tree_valid"]["shape"][0] != splits["valid"]["rows"]:
        raise ValueError("tree valid rows do not match valid groups")
    if arrays["tree_test"]["shape"][0] != splits["test"]["rows"]:
        raise ValueError("tree test rows do not match test groups")
    for split in ("train", "valid", "test"):
        if arrays[f"linear_{split}"]["shape"][0] != splits[split]["rows"]:
            raise ValueError(f"linear {split} rows do not match {split} groups")
    for split in ("train", "valid", "test"):
        expected_split = manifest.get("splits", {}).get(split, {})
        if int(expected_split.get("start", -1)) != splits[split]["time_min"]:
            raise ValueError(f"{split} manifest start disagrees with common time")
        if int(expected_split.get("stop", -1)) != splits[split]["time_max"] + 1:
            raise ValueError(f"{split} manifest stop disagrees with common time")
        if int(expected_split.get("rows", -1)) != splits[split]["rows"]:
            raise ValueError(f"{split} manifest rows disagree with group sizes")
    if arrays["sequence_mask"]["shape"] != [3603, 5282]:
        raise ValueError("sequence mask must cover all time/stock cells")

    result = CacheContract(manifest=manifest, arrays=arrays, splits=splits)
    print("processed cache contract: OK")
    print({k: {kk: vv for kk, vv in v.items() if kk in ("rows", "n_times", "time_min", "time_max")} for k, v in splits.items()})
    return result

In [11]:
def _check_vector(path: Path, expected: int, name: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(path)
    arr = np.load(path, allow_pickle=False)
    if arr.ndim != 1 or arr.size != expected:
        raise ValueError(f"{name}: shape {arr.shape}, expected ({expected},)")
    finite_array(arr, name)
    return {"path": str(path), "shape": list(arr.shape), "dtype": str(arr.dtype), "sha256": sha256_file(path)}


def _cache_npz(path: Path, expected: int, required: Sequence[str]) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as z:
        missing = [key for key in required if key not in z.files]
        if missing:
            raise KeyError(f"{path.name} missing {missing}")
        arrays = {}
        for key in required:
            arr = np.asarray(z[key])
            if arr.ndim != 1 or arr.size != expected:
                raise ValueError(f"{path.name}:{key} has shape {arr.shape}, expected ({expected},)")
            finite_array(arr, f"{path.name}:{key}")
            arrays[key] = arr.astype(np.float32, copy=False)
    return {"path": str(path), "sha256": sha256_file(path), "arrays": arrays}


def load_anchor_bundle(contract: CacheContract | None = None) -> dict[str, Any]:
    contract = contract or validate_processed_cache()
    train_groups = contract.splits["train"]["groups"]
    valid_groups = contract.splits["valid"]["groups"]
    test_groups = contract.splits["test"]["groups"]
    # exp_009 stores predictions for each fold's *validation interval*:
    # [2189,2432), [2432,2675), [2675,2918). The train group array
    # starts at absolute time 486, so convert absolute time to offset.
    fold_specs = {
        "fold_1": (486, 2189, 2432, "fold_1_3a8382e5ee27a3a1.npz"),
        "fold_2": (486, 2432, 2675, "fold_2_3a8382e5ee27a3a1.npz"),
        "fold_3": (486, 2675, 2918, "fold_3_3a8382e5ee27a3a1.npz"),
    }
    fold_cache = {}
    for name, (train_start, valid_start, valid_stop, filename) in fold_specs.items():
        train_origin = int(train_start)
        valid_lo = int(valid_start) - train_origin
        valid_hi = int(valid_stop) - train_origin
        expected = int(train_groups[valid_lo:valid_hi].sum())
        fold_cache[name] = _cache_npz(EXP009_CACHE / filename, expected, ("base_prediction", "recent_prediction_8", "recent_prediction_16"))
        fold_cache[name]["train_start"] = train_start
        fold_cache[name]["valid_start"] = valid_start
        fold_cache[name]["valid_stop"] = valid_stop
        fold_cache[name]["valid_lo"] = valid_lo
        fold_cache[name]["valid_hi"] = valid_hi

    official_path = EXP009_CACHE / "official_valid_3a8382e5ee27a3a1_r16.npz"
    official = _cache_npz(official_path, int(valid_groups.sum()), ("base_prediction", "expert_prediction"))
    test_raw_path = EXP009_CACHE / "raw_test_6099a59b9ae4f7b3.npy"
    test_raw = _check_vector(test_raw_path, int(test_groups.sum()), "raw_test_prediction")
    test_raw["array"] = np.load(test_raw_path, allow_pickle=False).astype(np.float32, copy=False)

    # Reproduce exp_009's selected blend on every available interval.
    for item in fold_cache.values():
        valid_groups = train_groups[item["valid_lo"] : item["valid_hi"]]
        item["blend_rank"] = group_rank((1.0 - EXP009_WEIGHT) * group_rank(item["arrays"]["base_prediction"], valid_groups) + EXP009_WEIGHT * group_rank(item["arrays"]["recent_prediction_16"], valid_groups), valid_groups)
    official_groups = contract.splits["valid"]["groups"]
    official["blend_rank"] = (
        (1.0 - EXP009_WEIGHT) * group_rank(official["arrays"]["base_prediction"], official_groups)
        + EXP009_WEIGHT * group_rank(official["arrays"]["expert_prediction"], official_groups)
    ).astype(np.float32)
    test_raw["rank"] = group_rank(test_raw["array"], test_groups)

    valid_grid = np.load(EXP009_DIR / "valid_prediction.npy", mmap_mode="r", allow_pickle=False)
    valid_times = np.load(CACHE_DIR / "common" / "valid_time.npy", mmap_mode="r", allow_pickle=False).astype(np.int64)
    valid_stocks = np.load(CACHE_DIR / "common" / "valid_stock.npy", mmap_mode="r", allow_pickle=False).astype(np.int64)
    valid_saved = np.asarray(valid_grid[valid_times - int(valid_times.min()), valid_stocks], dtype=np.float32)
    if not np.allclose(official["blend_rank"], valid_saved, atol=1e-7, rtol=1e-6):
        raise ValueError("exp_009 official valid cache does not reproduce valid_prediction.npy")

    test_grid = np.load(EXP009_DIR / "prediction.npy", mmap_mode="r", allow_pickle=False)
    test_times = np.load(CACHE_DIR / "common" / "test_time.npy", mmap_mode="r", allow_pickle=False).astype(np.int64)
    test_stocks = np.load(CACHE_DIR / "common" / "test_stock.npy", mmap_mode="r", allow_pickle=False).astype(np.int64)
    test_anchor = np.asarray(test_grid[test_times - int(test_times.min()), test_stocks], dtype=np.float32)
    finite_array(test_anchor, "exp009_test_anchor")
    test_raw["recent_rank"] = test_raw.pop("rank")
    test_raw["rank"] = test_anchor

    validation = {
        "mode": ANCHOR_MODE,
        "exp009_weight": EXP009_WEIGHT,
        "folds": {name: {"sha256": item["sha256"], "rows": int(item["blend_rank"].size), "finite": bool(np.isfinite(item["blend_rank"]).all())} for name, item in fold_cache.items()},
        "official_valid": {"sha256": official["sha256"], "rows": int(official["blend_rank"].size), "finite": bool(np.isfinite(official["blend_rank"]).all())},
        "test": {"sha256": test_raw["sha256"], "rows": int(test_raw["rank"].size), "finite": bool(np.isfinite(test_raw["rank"]).all())},
        "status": "cache_valid",
    }
    return {"folds": fold_cache, "official_valid": official, "test": test_raw, "validation": validation}


def rebuild_anchor_inference(contract: CacheContract | None = None, reason: str = "cache_invalid") -> dict[str, Any]:
    contract = contract or validate_processed_cache()
    groups = contract.splits["test"]["groups"]
    if not FINAL_SUBMISSION.exists():
        raise FileNotFoundError("cannot rebuild anchor: final_submission/prediction.npy is missing")
    grid = np.load(FINAL_SUBMISSION, allow_pickle=False)
    assert_prediction_grid(grid, groups, "final_submission/prediction.npy")
    test_time = np.load(CACHE_DIR / "common" / "test_time.npy", allow_pickle=False).astype(np.int64, copy=False)
    test_stock = np.load(CACHE_DIR / "common" / "test_stock.npy", allow_pickle=False).astype(np.int64, copy=False)
    local_time = test_time - int(test_time.min())
    if local_time.min() < 0 or local_time.max() >= grid.shape[0] or test_stock.min() < 0 or test_stock.max() >= grid.shape[1]:
        raise ValueError("final submission grid cannot be aligned to test common arrays")
    flat = grid[local_time, test_stock].astype(np.float32, copy=False)
    bundle = {"folds": {}, "official_valid": None, "test": {"array": flat, "rank": group_rank(flat, groups)}, "validation": {"mode": "rebuild_inference", "reason": reason, "status": "rebuilt_from_final_submission", "source_sha256": sha256_file(FINAL_SUBMISSION)}}
    return bundle


def load_anchor_or_rebuild(contract: CacheContract, require_oof: bool = False) -> dict[str, Any]:
    try:
        bundle = load_anchor_bundle(contract)
    except Exception as exc:
        if require_oof:
            raise RuntimeError(f"exp_009 OOF anchor validation failed; full training stopped safely: {type(exc).__name__}: {exc}") from exc
        if ANCHOR_MODE != "reuse_then_rebuild_inference":
            raise
        print(f"anchor cache validation failed; rebuilding inference anchor: {type(exc).__name__}: {exc}")
        bundle = rebuild_anchor_inference(contract, reason=f"{type(exc).__name__}: {exc}")
    json_dump(RUN_DIR / "anchor_validation.json", bundle["validation"])
    return bundle

In [12]:
class MaskedTemporalRankGLU(nn.Module):
    def __init__(self, current_dim: int = CURRENT_FEATURES, sequence_dim: int = SEQUENCE_FEATURES, hidden: int = 128, bottleneck: int = 128):
        super().__init__()
        self.current_encoder = nn.Sequential(
            nn.Linear(current_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
        )
        self.temporal_proj = nn.Linear(sequence_dim, hidden)
        self.temporal_conv = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 4 + 2, hidden),
            nn.GELU(),
            nn.LayerNorm(hidden),
        )
        self.score_norm = nn.LayerNorm(hidden)
        self.linear_score = nn.Linear(hidden, 1)
        self.value_path = nn.Linear(hidden, bottleneck)
        self.gate_path = nn.Linear(hidden, bottleneck)
        self.gated_score = nn.Linear(bottleneck, 1)
        self.gamma_raw = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))

    def forward(self, current: torch.Tensor, sequence: torch.Tensor, mask: torch.Tensor, anchor_z: torch.Tensor) -> torch.Tensor:
        current = current.float()
        sequence = sequence.float()
        mask = mask.float().clamp(0.0, 1.0)
        anchor_z = anchor_z.float().reshape(-1)
        if current.ndim != 2 or current.shape[1] != CURRENT_FEATURES:
            raise ValueError(f"current must be [N,{CURRENT_FEATURES}], got {tuple(current.shape)}")
        if sequence.ndim != 3 or sequence.shape[1:] != (WINDOW, SEQUENCE_FEATURES):
            raise ValueError(f"sequence must be [N,{WINDOW},{SEQUENCE_FEATURES}], got {tuple(sequence.shape)}")
        if sequence.shape[0] != current.shape[0] or mask.shape != sequence.shape[:2] or anchor_z.numel() != current.shape[0]:
            raise ValueError("mask/anchor batch shape mismatch")
        if not torch.isfinite(current).all() or not torch.isfinite(sequence).all() or not torch.isfinite(mask).all() or not torch.isfinite(anchor_z).all():
            raise ValueError("model inputs contain non-finite values")

        current_h = self.current_encoder(current)
        weights = mask.unsqueeze(-1)
        temporal = self.temporal_proj(sequence) * weights
        temporal = F.gelu(self.temporal_conv(temporal.transpose(1, 2))).transpose(1, 2) * weights
        denom = weights.sum(dim=1).clamp_min(1.0)
        mean = (temporal * weights).sum(dim=1) / denom
        has_valid = mask.sum(dim=1) > 0
        masked = temporal.masked_fill(weights <= 0, -torch.finfo(temporal.dtype).max)
        max_pool = masked.max(dim=1).values
        max_pool = torch.where(has_valid.unsqueeze(1), max_pool, torch.zeros_like(max_pool))
        positions = torch.arange(sequence.shape[1], device=sequence.device).unsqueeze(0).expand_as(mask)
        last_index = positions.masked_fill(mask <= 0, -1).max(dim=1).values.clamp_min(0)
        last = temporal[torch.arange(sequence.shape[0], device=sequence.device), last_index]
        last = torch.where(has_valid.unsqueeze(1), last, torch.zeros_like(last))
        coverage = mask.mean(dim=1) * 2.0 - 1.0
        fused = self.fusion(torch.cat([current_h, mean, max_pool, last, coverage.unsqueeze(1), anchor_z.unsqueeze(1)], dim=1))
        h = self.score_norm(fused)
        direct = self.linear_score(h).squeeze(1)
        gated = self.gated_score(self.value_path(h) * torch.sigmoid(self.gate_path(h))).squeeze(1)
        gamma = 0.25 * torch.tanh(self.gamma_raw)
        return (direct + gamma * gated).float()


def mean_group_pearson(pred: torch.Tensor, target: torch.Tensor, group_sizes: Sequence[int], eps: float = 1e-6) -> torch.Tensor:
    pred = pred.float().reshape(-1)
    target = target.float().reshape(-1)
    correlations = []
    offset = 0
    for size in group_sizes:
        size = int(size)
        p = pred[offset : offset + size]
        y = target[offset : offset + size]
        p = p - p.mean()
        y = y - y.mean()
        correlations.append((p * y).mean() / (p.square().mean().sqrt() * y.square().mean().sqrt() + eps))
        offset += size
    if offset != pred.numel():
        raise ValueError("correlation group mismatch")
    return torch.stack(correlations).mean()


def temporal_residual_loss(residual: torch.Tensor, anchor: torch.Tensor, target: torch.Tensor, group_sizes: Sequence[int], alpha: float = EARLY_STOP_ALPHA) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    residual = residual.float().reshape(-1)
    anchor_z = group_zscore(anchor.float().reshape(-1), group_sizes)
    target_z = group_zscore(target.float().reshape(-1), group_sizes)
    residual_z = group_zscore(residual, group_sizes)
    corrected = group_zscore(anchor_z + float(alpha) * residual_z, group_sizes)
    mse = F.mse_loss(corrected, target_z)
    ic_loss = 1.0 - mean_group_pearson(corrected, target_z, group_sizes)
    residual_loss = F.smooth_l1_loss(residual, target_z - anchor_z)
    total = mse + 0.10 * ic_loss + 0.25 * residual_loss
    return total, {"mse": mse.detach(), "ic_loss": ic_loss.detach(), "residual_loss": residual_loss.detach()}


def build_temporal_batch(split: str, time_index: int, stock_indices: np.ndarray | Sequence[int] | None = None, window: int = WINDOW) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if split not in {"train", "valid", "test"}:
        raise ValueError(split)
    tree = np.load(CACHE_DIR / "tree" / f"{split}_X.npy", mmap_mode="r", allow_pickle=False)
    common_time = np.load(CACHE_DIR / "common" / f"{split}_time.npy", mmap_mode="r", allow_pickle=False)
    common_stock = np.load(CACHE_DIR / "common" / f"{split}_stock.npy", mmap_mode="r", allow_pickle=False)
    seq = np.load(CACHE_DIR / "sequence" / "X.npy", mmap_mode="r", allow_pickle=False)
    seq_mask = np.load(CACHE_DIR / "sequence" / "mask_x.npy", mmap_mode="r", allow_pickle=False)
    group_sizes = np.load(CACHE_DIR / "common" / f"{split}_group_sizes.npy", allow_pickle=False).astype(np.int64)
    if time_index < 0 or time_index >= len(group_sizes):
        raise IndexError(time_index)
    offsets = np.concatenate([[0], np.cumsum(group_sizes)])
    lo, hi = int(offsets[time_index]), int(offsets[time_index + 1])
    rows = np.arange(lo, hi) if stock_indices is None else np.asarray(stock_indices, dtype=np.int64)
    if stock_indices is not None:
        rows = rows[(rows >= lo) & (rows < hi)]
    if rows.size == 0:
        raise ValueError("empty cross-section")
    stocks = common_stock[rows].astype(np.int64)
    times = common_time[rows].astype(np.int64)
    if not np.all(times == times[0]) or np.any(np.diff(stocks) <= 0):
        raise ValueError(f"{split} cross-section row order is not strictly (time, stock)")
    end_time = int(times[0])
    start_time = max(0, end_time - int(window) + 1)
    sequence_batch = np.asarray(seq[start_time : end_time + 1, stocks, :], dtype=np.float32).transpose(1, 0, 2)
    mask_batch = np.asarray(seq_mask[start_time : end_time + 1, stocks], dtype=np.float32).transpose(1, 0)
    current = np.asarray(tree[rows, :CURRENT_FEATURES], dtype=np.float32)
    if sequence_batch.shape[1] < window:
        pad = window - sequence_batch.shape[1]
        sequence_batch = np.pad(sequence_batch, ((0, 0), (pad, 0), (0, 0)), mode="constant")
        mask_batch = np.pad(mask_batch, ((0, 0), (pad, 0)), mode="constant")
    finite_array(current, f"{split}.current")
    finite_array(sequence_batch, f"{split}.sequence")
    finite_array(mask_batch, f"{split}.mask")
    if sequence_batch.shape != (rows.size, window, SEQUENCE_FEATURES) or mask_batch.shape != (rows.size, window):
        raise ValueError(f"{split} temporal batch shape mismatch")
    return current, sequence_batch, mask_batch, rows


def predict_full_cross_section(*args: Any, **kwargs: Any) -> np.ndarray:
    raise RuntimeError("v2 requires stats and anchor_z; use predict_split_by_time()")


def train_temporal_rankglu(*args: Any, **kwargs: Any) -> MaskedTemporalRankGLU:
    raise RuntimeError("v2 uses staged train_stage(); call run_full() or run_preflight()")


def oof_anchor_interval(anchor_bundle: Mapping[str, Any], fold_name: str, contract: CacheContract) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    start_time, stop_time = OOF_INTERVALS[fold_name]
    groups = contract.splits["train"]["groups"][start_time - TRAIN_TIME_ORIGIN : stop_time - TRAIN_TIME_ORIGIN]
    item = anchor_bundle["folds"][fold_name]
    base = group_rank(item["arrays"]["base_prediction"], groups)
    recent = group_rank(item["arrays"]["recent_prediction_16"], groups)
    anchor = ((1.0 - EXP009_WEIGHT) * base + EXP009_WEIGHT * recent).astype(np.float32)
    offsets = np.concatenate([[0], np.cumsum(contract.splits["train"]["groups"])])
    row_lo = int(offsets[start_time - TRAIN_TIME_ORIGIN])
    row_hi = int(offsets[stop_time - TRAIN_TIME_ORIGIN])
    if anchor.size != row_hi - row_lo or anchor.size != int(groups.sum()):
        raise ValueError(f"{fold_name} OOF anchor alignment failed")
    return anchor, groups, np.arange(row_lo, row_hi, dtype=np.int64)


def slice_oof_fold(anchor_bundle: Mapping[str, Any], fold_name: str, contract: CacheContract, start_time: int, stop_time: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    fold_start, fold_stop = OOF_INTERVALS[fold_name]
    if start_time < fold_start or stop_time > fold_stop or start_time >= stop_time:
        raise ValueError(f"invalid {fold_name} slice [{start_time},{stop_time})")
    full_anchor, full_groups, full_rows = oof_anchor_interval(anchor_bundle, fold_name, contract)
    local_lo = start_time - fold_start
    local_hi = stop_time - fold_start
    offsets = np.concatenate([[0], np.cumsum(full_groups)])
    row_lo, row_hi = int(offsets[local_lo]), int(offsets[local_hi])
    return full_anchor[row_lo:row_hi], full_groups[local_lo:local_hi], full_rows[row_lo:row_hi]


def interval_schedule(contract: CacheContract, anchor_bundle: Mapping[str, Any], intervals: Sequence[tuple[str, int, int]]) -> list[dict[str, Any]]:
    schedule = []
    for fold_name, start_time, stop_time in intervals:
        fold_anchor, _, fold_rows = slice_oof_fold(anchor_bundle, fold_name, contract, start_time, stop_time)
        cursor = 0
        for absolute_time in range(start_time, stop_time):
            local_time = absolute_time - TRAIN_TIME_ORIGIN
            size = int(contract.splits["train"]["groups"][local_time])
            rows = fold_rows[cursor : cursor + size]
            schedule.append({"fold": fold_name, "time": absolute_time, "local_time": local_time, "rows": rows, "anchor": fold_anchor[cursor : cursor + size]})
            cursor += size
        if cursor != len(fold_rows):
            raise ValueError(f"{fold_name} schedule alignment failed")
    return schedule


def normalization_fingerprint(intervals: Sequence[tuple[str, int, int]], contract: CacheContract) -> str:
    payload = {
        "intervals": [list(x) for x in intervals],
        "manifest_sha256": sha256_file(CACHE_DIR / "manifest.json"),
        "window": WINDOW,
        "current_features": CURRENT_FEATURES,
        "sequence_features": SEQUENCE_FEATURES,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


def compute_normalization_stats(contract: CacheContract, anchor_bundle: Mapping[str, Any], intervals: Sequence[tuple[str, int, int]]) -> dict[str, Any]:
    schedule = interval_schedule(contract, anchor_bundle, intervals)
    current_sum = np.zeros(CURRENT_FEATURES, dtype=np.float64)
    current_sq = np.zeros(CURRENT_FEATURES, dtype=np.float64)
    sequence_sum = np.zeros(SEQUENCE_FEATURES, dtype=np.float64)
    sequence_sq = np.zeros(SEQUENCE_FEATURES, dtype=np.float64)
    current_count = 0
    sequence_count = 0
    for item in schedule:
        current, sequence, mask, _ = build_temporal_batch("train", item["local_time"])
        current64 = current.astype(np.float64, copy=False)
        current_sum += current64.sum(axis=0)
        current_sq += np.square(current64).sum(axis=0)
        current_count += current.shape[0]
        mask64 = mask.astype(np.float64, copy=False)[..., None]
        sequence64 = sequence.astype(np.float64, copy=False)
        sequence_sum += (sequence64 * mask64).sum(axis=(0, 1))
        sequence_sq += (np.square(sequence64) * mask64).sum(axis=(0, 1))
        sequence_count += int(mask.sum())
    if current_count <= 0 or sequence_count <= 0:
        raise ValueError("normalization received no valid rows")
    current_mean = current_sum / current_count
    sequence_mean = sequence_sum / sequence_count
    current_std = np.sqrt(np.maximum(current_sq / current_count - np.square(current_mean), 1e-6))
    sequence_std = np.sqrt(np.maximum(sequence_sq / sequence_count - np.square(sequence_mean), 1e-6))
    return {
        "current_mean": current_mean.astype(np.float32),
        "current_std": current_std.astype(np.float32),
        "sequence_mean": sequence_mean.astype(np.float32),
        "sequence_std": sequence_std.astype(np.float32),
        "current_count": int(current_count),
        "sequence_count": int(sequence_count),
        "fingerprint": normalization_fingerprint(intervals, contract),
        "intervals": [tuple(x) for x in intervals],
    }


def normalize_batch(current: np.ndarray, sequence: np.ndarray, mask: np.ndarray, stats: Mapping[str, Any]) -> tuple[np.ndarray, np.ndarray]:
    current_norm = np.clip((current - stats["current_mean"]) / stats["current_std"], -6.0, 6.0).astype(np.float32)
    sequence_norm = np.clip((sequence - stats["sequence_mean"]) / stats["sequence_std"], -6.0, 6.0).astype(np.float32)
    sequence_norm *= mask[..., None].astype(np.float32)
    finite_array(current_norm, "normalized_current")
    finite_array(sequence_norm, "normalized_sequence")
    return current_norm, sequence_norm


def save_normalization_stats(path: Path, stats: Mapping[str, Any]) -> None:
    atomic_npz_save(
        path,
        current_mean=stats["current_mean"], current_std=stats["current_std"],
        sequence_mean=stats["sequence_mean"], sequence_std=stats["sequence_std"],
        current_count=np.int64(stats["current_count"]), sequence_count=np.int64(stats["sequence_count"]),
        fingerprint=np.asarray(stats["fingerprint"]), intervals=np.asarray(stats["intervals"], dtype=object),
    )


def config_fingerprint(
    stage: str, seed: int, intervals: Sequence[tuple[str, int, int]], fixed_epochs: int | None,
    stats: Mapping[str, Any], validation_intervals: Sequence[tuple[str, int, int]] | None = None,
) -> str:
    payload = {
        "stage": stage, "seed": int(seed), "intervals": [list(x) for x in intervals],
        "validation_intervals": [list(x) for x in (validation_intervals or [])],
        "fixed_epochs": fixed_epochs, "max_epochs": MAX_EPOCHS, "min_epochs": MIN_EPOCHS,
        "patience": EARLY_STOP_PATIENCE, "lr": LEARNING_RATE, "min_lr": MIN_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "grad_clip": GRAD_CLIP, "selection_alpha": EARLY_STOP_ALPHA,
        "normalization": stats["fingerprint"], "model": "masked_temporal_rankglu_128_v2",
        "loss": "mse_corrected+0.1_ic+0.25_smoothl1_raw_residual",
        "precision": PRIMARY_PRECISION, "stability_policy": STABILITY_POLICY,
        "torch": torch.__version__, "version": CHECKPOINT_SCHEMA_VERSION,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


def capture_rng_state() -> dict[str, Any]:
    return {
        "python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state: Mapping[str, Any]) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def tensors_finite(value: Any) -> bool:
    if torch.is_tensor(value):
        return bool(torch.isfinite(value).all()) if value.is_floating_point() else True
    if isinstance(value, Mapping):
        return all(tensors_finite(item) for item in value.values())
    if isinstance(value, (list, tuple)):
        return all(tensors_finite(item) for item in value)
    return True


def reject_checkpoint(path: Path, reason: str) -> None:
    if not path.exists():
        return
    rejected = path.with_name(f"{path.name}.rejected_{time.time_ns()}")
    os.replace(path, rejected)
    json_dump(rejected.with_suffix(rejected.suffix + ".json"), {
        "status": "checkpoint_rejected", "original": str(path), "retained_as": str(rejected), "reason": reason,
    })


def load_checkpoint_checked(path: Path, fingerprint: str, required: Sequence[str]) -> dict[str, Any]:
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        missing = [key for key in required if key not in checkpoint]
        if missing:
            raise ValueError(f"missing fields: {missing}")
        if checkpoint["fingerprint"] != fingerprint:
            raise ValueError("fingerprint mismatch")
        if not tensors_finite(checkpoint["state_dict"]):
            raise ValueError("model parameters contain non-finite values")
        return checkpoint
    except Exception as exc:
        reject_checkpoint(path, f"{type(exc).__name__}: {exc}")
        raise RuntimeError(f"checkpoint rejected: {path.name}: {exc}") from exc


def precision_context(precision: str):
    enabled = DEVICE.type == "cuda" and precision in {"bf16", "fp16"}
    dtype = torch.bfloat16 if precision == "bf16" else torch.float16
    return torch.autocast(device_type=DEVICE.type, dtype=dtype, enabled=enabled)


def finite_diagnostics(tensor: torch.Tensor | None) -> dict[str, Any]:
    if tensor is None:
        return {"present": False}
    detached = tensor.detach().float()
    finite = torch.isfinite(detached)
    values = detached[finite]
    return {
        "present": True, "shape": list(detached.shape), "finite": bool(finite.all()),
        "nonfinite_count": int((~finite).sum().cpu()),
        "finite_min": float(values.min().cpu()) if values.numel() else None,
        "finite_max": float(values.max().cpu()) if values.numel() else None,
    }


def write_numerical_failure(context: Mapping[str, Any], attempts: Sequence[Mapping[str, Any]]) -> Path:
    path = RUN_DIR / "failures" / f"{context['stage']}_seed_{context['seed']}_epoch_{context['epoch']}_time_{context['time']}.json"
    json_dump(path, {"status": "numerical_failure", **dict(context), "attempts": list(attempts)})
    return path


def stable_training_step(
    model: nn.Module, optimizer: torch.optim.Optimizer, scaler: torch.cuda.amp.GradScaler,
    current_t: torch.Tensor, sequence_t: torch.Tensor, mask_t: torch.Tensor,
    anchor_z_t: torch.Tensor, anchor_t: torch.Tensor, target_t: torch.Tensor,
    group_sizes: Sequence[int], context: Mapping[str, Any], fault_injection: str | None = None,
) -> tuple[float, dict[str, Any]]:
    attempts = []
    precisions = [PRIMARY_PRECISION] + ([] if PRIMARY_PRECISION == "fp32" else ["fp32"])
    for attempt_index, precision in enumerate(precisions):
        optimizer.zero_grad(set_to_none=True)
        residual = None
        loss = None
        scaler_unscaled = False
        try:
            with precision_context(precision):
                residual = model(current_t, sequence_t, mask_t, anchor_z_t)
            with torch.autocast(device_type=DEVICE.type, enabled=False):
                loss, _ = temporal_residual_loss(residual.float(), anchor_t, target_t, group_sizes, EARLY_STOP_ALPHA)
                if fault_injection == "primary_nonfinite" and attempt_index == 0:
                    loss = loss * torch.tensor(float("nan"), device=loss.device)
                if fault_injection == "all_nonfinite":
                    loss = loss * torch.tensor(float("nan"), device=loss.device)
            if not torch.isfinite(residual).all() or not torch.isfinite(loss):
                raise FloatingPointError("non-finite model output or loss")
            use_scaler = precision == "fp16" and scaler.is_enabled()
            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                scaler_unscaled = True
            else:
                loss.backward()
            gradients_finite = all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
            if not gradients_finite:
                raise FloatingPointError("non-finite gradients")
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP, error_if_nonfinite=True)
            if not torch.isfinite(grad_norm):
                raise FloatingPointError("non-finite gradient norm")
            if use_scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            return float(loss.detach().cpu()), {
                "precision": precision, "retried_fp32": attempt_index == 1,
                "grad_norm": float(grad_norm.detach().cpu()), "failed_attempts": attempts,
            }
        except (FloatingPointError, RuntimeError) as exc:
            if isinstance(exc, RuntimeError) and "non-finite" not in str(exc).lower():
                optimizer.zero_grad(set_to_none=True)
                raise
            gradient_nonfinite = sum(
                int((~torch.isfinite(parameter.grad)).sum().detach().cpu())
                for parameter in model.parameters() if parameter.grad is not None
            )
            attempts.append({
                "precision": precision, "error": f"{type(exc).__name__}: {exc}",
                "output": finite_diagnostics(residual), "loss": finite_diagnostics(loss),
                "gradient_nonfinite_count": gradient_nonfinite,
            })
            optimizer.zero_grad(set_to_none=True)
            if scaler_unscaled:
                # Reset GradScaler's per-optimizer stage after a rejected
                # FP16 step. No optimizer update has occurred.
                scaler.update()
            if attempt_index + 1 < len(precisions):
                continue
            failure_path = write_numerical_failure(context, attempts)
            raise RuntimeError(f"non-finite training step; diagnostics: {failure_path}") from exc
    raise AssertionError("unreachable training precision loop")


def epoch_lr(epoch_index: int, total_epochs: int) -> float:
    if epoch_index == 0:
        return 0.25
    progress = (epoch_index - 1) / max(total_epochs - 2, 1)
    minimum_ratio = MIN_LEARNING_RATE / LEARNING_RATE
    return minimum_ratio + (1.0 - minimum_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))


def evaluate_model_schedule(model: nn.Module, schedule: Sequence[Mapping[str, Any]], stats: Mapping[str, Any], target: np.ndarray, alpha: float = EARLY_STOP_ALPHA) -> float:
    anchor_parts, residual_parts, target_parts, groups = [], [], [], []
    model.eval()
    with torch.no_grad():
        for item in schedule:
            current, sequence, mask, rows = build_temporal_batch("train", item["local_time"])
            current, sequence = normalize_batch(current, sequence, mask, stats)
            anchor = np.asarray(item["anchor"], dtype=np.float32)
            anchor_z = group_zscore_np(anchor, [len(anchor)])
            residual = model(
                torch.from_numpy(current).to(DEVICE), torch.from_numpy(sequence).to(DEVICE),
                torch.from_numpy(mask).to(DEVICE), torch.from_numpy(anchor_z).to(DEVICE),
            ).cpu().numpy().astype(np.float32)
            anchor_parts.append(anchor); residual_parts.append(residual); target_parts.append(np.asarray(target[rows], dtype=np.float32)); groups.append(len(rows))
    corrected = residual_correct(np.concatenate(anchor_parts), np.concatenate(residual_parts), np.asarray(groups), alpha)
    return evaluate_rankic(corrected, np.concatenate(target_parts), groups)


def train_stage(
    stage: str, seed: int, contract: CacheContract, anchor_bundle: Mapping[str, Any],
    train_intervals: Sequence[tuple[str, int, int]], stats: Mapping[str, Any],
    fixed_epochs: int | None = None, validation_intervals: Sequence[tuple[str, int, int]] | None = None,
    history: list[dict[str, Any]] | None = None, resume: bool = RESUME,
) -> tuple[MaskedTemporalRankGLU, int, float]:
    history = history if history is not None else []
    set_seed(seed)
    model = MaskedTemporalRankGLU().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_epochs = int(fixed_epochs or MAX_EPOCHS)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda e: epoch_lr(e, total_epochs))
    train_schedule = interval_schedule(contract, anchor_bundle, train_intervals)
    valid_schedule = interval_schedule(contract, anchor_bundle, validation_intervals) if validation_intervals else []
    y = np.load(CACHE_DIR / "common" / "train_y.npy", mmap_mode="r", allow_pickle=False)
    fingerprint = config_fingerprint(stage, seed, train_intervals, fixed_epochs, stats, validation_intervals)
    model_dir = RUN_DIR / "models_v4" / stage / f"seed_{seed}" / fingerprint[:16]
    model_dir.mkdir(parents=True, exist_ok=True)
    resume_path = model_dir / "resume.pt"
    best_path = model_dir / "best.pt"
    start_epoch, best_metric, wait = 0, -float("inf"), 0
    best_state = copy.deepcopy(model.state_dict())
    stopped_early = False
    stage_history: list[dict[str, Any]] = []
    scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
    if resume and resume_path.exists():
        try:
            checkpoint = load_checkpoint_checked(resume_path, fingerprint, (
                "state_dict", "optimizer", "scheduler", "epoch", "best_metric", "wait",
                "stopped_early", "fingerprint", "history", "rng_state",
            ))
            if best_path.exists():
                best_checkpoint = load_checkpoint_checked(best_path, fingerprint, ("state_dict", "epoch", "metric", "fingerprint"))
                if int(best_checkpoint["epoch"]) > int(checkpoint["epoch"]):
                    raise RuntimeError("best checkpoint is newer than the last complete resume epoch")
                if abs(float(best_checkpoint["metric"]) - float(checkpoint["best_metric"])) > 1e-12:
                    raise RuntimeError("best and resume checkpoint metrics disagree")
                best_state = best_checkpoint["state_dict"]
            elif np.isfinite(float(checkpoint["best_metric"])):
                raise RuntimeError("matching resume checkpoint has no best checkpoint")
            model.load_state_dict(checkpoint["state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            if USE_GRAD_SCALER and checkpoint.get("scaler"):
                scaler.load_state_dict(checkpoint["scaler"])
            if not tensors_finite(optimizer.state_dict()):
                raise RuntimeError("optimizer state contains non-finite values")
            start_epoch = int(checkpoint["epoch"]) + 1
            best_metric = float(checkpoint["best_metric"])
            wait = int(checkpoint["wait"])
            stopped_early = bool(checkpoint.get("stopped_early", False))
            stage_history = list(checkpoint.get("history", []))
            history.extend(stage_history)
            restore_rng_state(checkpoint["rng_state"])
            print({"stage": stage, "seed": seed, "resume_epoch": start_epoch})
        except Exception as exc:
            reject_checkpoint(resume_path, f"resume validation failed: {exc}")
            reject_checkpoint(best_path, f"paired resume validation failed: {exc}")
            set_seed(seed)
            model = MaskedTemporalRankGLU().to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda e: epoch_lr(e, total_epochs))
            scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
            start_epoch, best_metric, wait, stopped_early, stage_history = 0, -float("inf"), 0, False, []
            best_state = copy.deepcopy(model.state_dict())
            print({"stage": stage, "seed": seed, "checkpoint_ignored": str(exc)})
    last_epoch = start_epoch - 1
    epoch_range = range(start_epoch, total_epochs) if not stopped_early else range(0)
    for epoch in epoch_range:
        model.train()
        losses, fp32_retries = [], 0
        epoch_learning_rate = float(optimizer.param_groups[0]["lr"])
        for item in train_schedule:
            current, sequence, mask, rows = build_temporal_batch("train", item["local_time"])
            current, sequence = normalize_batch(current, sequence, mask, stats)
            anchor = np.asarray(item["anchor"], dtype=np.float32)
            anchor_z = group_zscore_np(anchor, [len(anchor)])
            loss_value, step_info = stable_training_step(
                model, optimizer, scaler,
                torch.from_numpy(current).to(DEVICE), torch.from_numpy(sequence).to(DEVICE),
                torch.from_numpy(mask).to(DEVICE), torch.from_numpy(anchor_z).to(DEVICE),
                torch.from_numpy(anchor).to(DEVICE),
                torch.from_numpy(np.asarray(y[rows], dtype=np.float32)).to(DEVICE), [len(rows)],
                {"stage": stage, "seed": int(seed), "epoch": epoch + 1, "fold": item["fold"], "time": int(item["time"])},
            )
            losses.append(loss_value)
            fp32_retries += int(step_info["retried_fp32"])
        metric = evaluate_model_schedule(model, valid_schedule, stats, y) if valid_schedule else -float(np.mean(losses))
        # Early-stop stages keep the best validation checkpoint. Fixed-epoch
        # stages deliberately keep the last epoch selected by the protocol.
        improved = (not validation_intervals) or metric > best_metric + 1e-7
        if improved:
            best_metric, wait = metric, 0
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            atomic_torch_save({
                "state_dict": best_state, "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "metric": metric,
                "fingerprint": fingerprint, "normalization": stats,
            }, best_path)
        else:
            wait += 1
        should_stop = bool(validation_intervals and epoch + 1 >= MIN_EPOCHS and wait >= EARLY_STOP_PATIENCE)
        scheduler.step()
        history_row = {"stage": stage, "seed": seed, "epoch": epoch + 1, "train_loss": float(np.mean(losses)), "selection_metric": metric, "best_metric": best_metric, "lr": epoch_learning_rate, "improved": improved, "primary_precision": PRIMARY_PRECISION, "fp32_retries": fp32_retries}
        history.append(history_row)
        stage_history.append(history_row)
        atomic_torch_save({
            "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "epoch": epoch, "best_metric": best_metric, "wait": wait,
            "stopped_early": should_stop, "fingerprint": fingerprint,
            "normalization_fingerprint": stats["fingerprint"], "normalization": stats,
            "history": stage_history, "rng_state": capture_rng_state(),
        }, resume_path)
        print(history_row)
        last_epoch = epoch
        if should_stop:
            break
    if not best_path.exists() or not np.isfinite(best_metric):
        raise RuntimeError(f"{stage} seed {seed} has no valid best checkpoint")
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    best_epoch = int(load_checkpoint_checked(best_path, fingerprint, ("state_dict", "epoch", "metric", "fingerprint"))["epoch"] + 1)
    return model, best_epoch, best_metric


def predict_split_by_time(
    model: nn.Module, split: str, contract: CacheContract, stats: Mapping[str, Any],
    anchor: np.ndarray, start_time: int | None = None, stop_time: int | None = None,
) -> np.ndarray:
    groups = contract.splits[split]["groups"]
    split_origin = contract.splits[split]["time_min"]
    first = 0 if start_time is None else int(start_time) - split_origin
    last = len(groups) if stop_time is None else int(stop_time) - split_origin
    if first < 0 or last > len(groups) or first >= last:
        raise ValueError(f"invalid {split} prediction interval: {start_time}, {stop_time}")
    expected = int(groups[first:last].sum())
    anchor = np.asarray(anchor, dtype=np.float32).reshape(-1)
    if anchor.size != expected:
        raise ValueError(f"{split} anchor length mismatch: {anchor.size} != {expected}")
    outputs = []
    cursor = 0
    model.eval()
    with torch.no_grad():
        for time_index in range(first, last):
            current, sequence, mask, _ = build_temporal_batch(split, time_index)
            current, sequence = normalize_batch(current, sequence, mask, stats)
            size = current.shape[0]
            anchor_block = anchor[cursor : cursor + size]
            anchor_z = group_zscore_np(anchor_block, [size])
            residual = model(
                torch.from_numpy(current).to(DEVICE),
                torch.from_numpy(sequence).to(DEVICE),
                torch.from_numpy(mask).to(DEVICE),
                torch.from_numpy(anchor_z).to(DEVICE),
            ).cpu().numpy().astype(np.float32)
            outputs.append(residual)
            cursor += size
    if cursor != expected:
        raise ValueError(f"{split} prediction cursor mismatch: {cursor} != {expected}")
    residual = np.concatenate(outputs)
    finite_array(residual, f"{split}_residual")
    return residual


def group_zscore_np(values: np.ndarray, group_sizes: Sequence[int], eps: float = 1e-6) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    groups = np.asarray(group_sizes, dtype=np.int64)
    if values.size != int(groups.sum()):
        raise ValueError("zscore group mismatch")
    out = np.empty_like(values)
    offset = 0
    for size in groups:
        size = int(size)
        block = values[offset : offset + size]
        out[offset : offset + size] = (block - block.mean()) / max(float(block.std()), eps)
        offset += size
    return out


def residual_correct(anchor: np.ndarray, residual: np.ndarray, groups: Sequence[int], alpha: float) -> np.ndarray:
    anchor = np.asarray(anchor, dtype=np.float32).reshape(-1)
    residual = np.asarray(residual, dtype=np.float32).reshape(-1)
    groups = np.asarray(groups, dtype=np.int64)
    if anchor.size != residual.size or anchor.size != int(groups.sum()):
        raise ValueError("residual correction length mismatch")
    if float(alpha) == 0.0:
        return anchor.copy()
    corrected = group_zscore_np(anchor, groups) + float(alpha) * group_zscore_np(residual, groups)
    return group_rank(corrected, groups)


def save_sparse_grid(
    values: np.ndarray, split: str, contract: CacheContract, filename: str, fill_value: float = 0.5,
) -> Path:
    times = np.load(CACHE_DIR / "common" / f"{split}_time.npy", allow_pickle=False).astype(np.int64)
    stocks = np.load(CACHE_DIR / "common" / f"{split}_stock.npy", allow_pickle=False).astype(np.int64)
    groups = contract.splits[split]["groups"]
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    if values.size != times.size or values.size != int(groups.sum()):
        raise ValueError("grid values/common/group length mismatch")
    finite_array(values, f"{filename}.values")
    dense = np.full((len(groups), 5282), np.float32(fill_value), dtype=np.float32)
    local_times = times - int(times.min())
    dense[local_times, stocks] = values
    path = RUN_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("wb") as handle:
        np.save(handle, dense)
    os.replace(temporary, path)
    saved = np.load(path, mmap_mode="r", allow_pickle=False)
    if saved.shape != dense.shape or saved.dtype != np.float32 or not np.isfinite(saved).all():
        raise RuntimeError(f"saved grid validation failed: {path}")
    if not np.array_equal(np.asarray(saved[local_times, stocks]), values):
        raise RuntimeError(f"saved grid values changed during write: {path}")
    return path


In [13]:
def evaluate_rankic(pred: np.ndarray, target: np.ndarray, group_sizes: Sequence[int]) -> float:
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    target = np.asarray(target, dtype=np.float32).reshape(-1)
    groups = np.asarray(group_sizes, dtype=np.int64)
    if pred.size != target.size or pred.size != int(groups.sum()):
        raise ValueError("RankIC length mismatch")
    scores, offset = [], 0
    for size in groups:
        size = int(size)
        p = rankdata(pred[offset : offset + size], method="average")
        y = rankdata(target[offset : offset + size], method="average")
        if p.std() > 0 and y.std() > 0:
            scores.append(float(np.corrcoef(p, y)[0, 1]))
        offset += size
    return float(np.mean(scores)) if scores else 0.0


def mean_cross_sectional_rank_correlation(left: np.ndarray, right: np.ndarray, groups: Sequence[int]) -> float:
    return evaluate_rankic(left, right, groups)


def write_csv(path: Path, rows: Sequence[Mapping[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)
    os.replace(temporary, path)


def validate_full_outputs() -> dict[str, dict[str, Any]]:
    required = [
        RUN_DIR / "prediction.npy", RUN_DIR / "candidate_prediction.npy", RUN_DIR / "anchor_prediction.npy",
        RUN_DIR / "expert_residual.npy", RUN_DIR / "valid_prediction.npy", RUN_DIR / "metrics.json",
        RUN_DIR / "submission_choice.json", RUN_DIR / "epoch_history.csv", RUN_DIR / "normalization_stats.npz",
    ]
    missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
    if missing:
        raise RuntimeError(f"full run output contract failed: {missing}")
    return {path.name: {"bytes": path.stat().st_size, "sha256": sha256_file(path)} for path in required}


def ensemble_standardized(residuals: Sequence[np.ndarray], groups: Sequence[int]) -> np.ndarray:
    return np.mean(np.stack([group_zscore_np(x, groups) for x in residuals]), axis=0).astype(np.float32)


def select_alpha(anchor: np.ndarray, residual: np.ndarray, target: np.ndarray, groups: Sequence[int]) -> tuple[float, list[dict[str, Any]]]:
    anchor_ic = evaluate_rankic(anchor, target, groups)
    rows = []
    for alpha in DIAGNOSTIC_NEGATIVE_ALPHAS + RESIDUAL_ALPHAS:
        prediction = residual_correct(anchor, residual, groups, alpha)
        score = evaluate_rankic(prediction, target, groups)
        rows.append({"alpha": float(alpha), "eligible": bool(alpha >= 0), "rankic": score, "delta": score - anchor_ic})
    eligible = [row for row in rows if row["eligible"]]
    best_delta = max(row["delta"] for row in eligible)
    tolerance = 0.05 * abs(best_delta)
    near = [row for row in eligible if row["delta"] >= best_delta - tolerance]
    return float(min(near, key=lambda row: row["alpha"])["alpha"]), rows


def rankic_diagnostics(pred: np.ndarray, target: np.ndarray, groups: Sequence[int]) -> dict[str, float]:
    pred = np.asarray(pred, dtype=np.float32).reshape(-1)
    target = np.asarray(target, dtype=np.float32).reshape(-1)
    scores, offset = [], 0
    for raw_size in groups:
        size = int(raw_size)
        p = rankdata(pred[offset : offset + size], method="average")
        y = rankdata(target[offset : offset + size], method="average")
        scores.append(float(np.corrcoef(p, y)[0, 1]) if p.std() > 0 and y.std() > 0 else np.nan)
        offset += size
    values = np.asarray(scores, dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise ValueError("no finite RankIC cross-sections")
    quarters = np.array_split(finite, 4)
    return {
        "mean": float(finite.mean()), "second_half": float(finite[finite.size // 2 :].mean()),
        "worst_quarter": float(min(float(part.mean()) for part in quarters if part.size)),
    }


def predict_oof_interval(model: nn.Module, fold_name: str, contract: CacheContract, anchor_bundle: Mapping[str, Any], stats: Mapping[str, Any]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    start, stop = OOF_INTERVALS[fold_name]
    anchor, groups, rows = oof_anchor_interval(anchor_bundle, fold_name, contract)
    residual = predict_split_by_time(model, "train", contract, stats, anchor, start, stop)
    return anchor, residual, groups, rows


def smoke_model_checks() -> dict[str, Any]:
    set_seed(SEEDS[0])
    n = 64
    model = MaskedTemporalRankGLU().to(DEVICE)
    current = torch.randn(n, CURRENT_FEATURES, device=DEVICE)
    sequence = torch.randn(n, WINDOW, SEQUENCE_FEATURES, device=DEVICE)
    mask = (torch.rand(n, WINDOW, device=DEVICE) > 0.15).float(); mask[:, -1] = 1
    anchor = torch.randn(n, device=DEVICE)
    target = torch.randn(n, device=DEVICE)
    residual = model(current, sequence, mask, anchor)
    loss, parts = temporal_residual_loss(residual, anchor, target, [n])
    loss.backward()
    return {"shape": list(residual.shape), "loss": float(loss.detach().cpu()), "parts": {k: float(v.cpu()) for k, v in parts.items()}, "gradient_finite": bool(all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()))}


def synthetic_stability_checks() -> dict[str, Any]:
    set_seed(20260812)
    n = 32
    current = torch.randn(n, CURRENT_FEATURES, device=DEVICE)
    sequence = torch.randn(n, WINDOW, SEQUENCE_FEATURES, device=DEVICE)
    mask = torch.ones(n, WINDOW, device=DEVICE)
    anchor = torch.randn(n, device=DEVICE)
    anchor_z = group_zscore(anchor, [n])
    target = torch.randn(n, device=DEVICE)

    def make_training_objects():
        local_model = MaskedTemporalRankGLU().to(DEVICE)
        local_optimizer = torch.optim.AdamW(local_model.parameters(), lr=1e-4)
        local_scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
        return local_model, local_optimizer, local_scaler

    model, optimizer, scaler = make_training_objects()
    _, normal = stable_training_step(
        model, optimizer, scaler, current, sequence, mask, anchor_z, anchor, target, [n],
        {"stage": "smoke_normal", "seed": 20260812, "epoch": 1, "fold": "synthetic", "time": 0},
    )
    if normal["retried_fp32"] or not tensors_finite(model.state_dict()):
        raise AssertionError("normal synthetic training step failed")

    model, optimizer, scaler = make_training_objects()
    _, retry = stable_training_step(
        model, optimizer, scaler, current, sequence, mask, anchor_z, anchor, target, [n],
        {"stage": "smoke_retry", "seed": 20260812, "epoch": 1, "fold": "synthetic", "time": 1},
        fault_injection="primary_nonfinite",
    )
    optimizer_steps = [int(state["step"].item()) for state in optimizer.state.values() if "step" in state]
    if PRIMARY_PRECISION != "fp32" and (not retry["retried_fp32"] or set(optimizer_steps) != {1}):
        raise AssertionError("AMP-to-FP32 retry did not produce exactly one optimizer step")

    with tempfile.TemporaryDirectory(prefix="exp014_v2_smoke_") as temp_dir:
        namespace = globals()
        original_run_dir = namespace["RUN_DIR"]
        namespace["RUN_DIR"] = Path(temp_dir)
        try:
            model, optimizer, scaler = make_training_objects()
            before = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            try:
                stable_training_step(
                    model, optimizer, scaler, current, sequence, mask, anchor_z, anchor, target, [n],
                    {"stage": "smoke_failure", "seed": 20260812, "epoch": 3, "fold": "synthetic", "time": 7},
                    fault_injection="all_nonfinite",
                )
                raise AssertionError("all-nonfinite fault injection unexpectedly succeeded")
            except RuntimeError as exc:
                if "diagnostics:" not in str(exc):
                    raise
            diagnostics = list((Path(temp_dir) / "failures").glob("*.json"))
            unchanged = all(torch.equal(before[key], model.state_dict()[key].detach().cpu()) for key in before)
            if len(diagnostics) != 1 or not unchanged or optimizer.state:
                raise AssertionError("failed step changed parameters or did not write one diagnostic")
            failure = json.loads(diagnostics[0].read_text(encoding="utf-8"))
            if failure.get("time") != 7 or len(failure.get("attempts", [])) != (1 if PRIMARY_PRECISION == "fp32" else 2):
                raise AssertionError("numerical failure diagnostic is incomplete")
        finally:
            namespace["RUN_DIR"] = original_run_dir
    return {
        "primary_precision": PRIMARY_PRECISION, "normal_step_finite": True,
        "fp32_retry_exercised": bool(retry["retried_fp32"]), "retry_optimizer_steps": sorted(set(optimizer_steps)),
        "dual_failure_diagnostic": True, "failed_step_left_model_unchanged": True,
    }


def synthetic_checkpoint_checks() -> dict[str, Any]:
    set_seed(314159)
    model = MaskedTemporalRankGLU().cpu()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda _: 1.0)
    fingerprint = "synthetic-checkpoint-v4"
    payload = {
        "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
        "scaler": {}, "epoch": 2, "best_metric": 0.125, "wait": 0, "stopped_early": False,
        "fingerprint": fingerprint, "history": [{"epoch": 3}], "rng_state": capture_rng_state(),
    }
    with tempfile.TemporaryDirectory(prefix="exp014_v2_checkpoint_") as temp_dir:
        path = Path(temp_dir) / "resume.pt"
        atomic_torch_save(payload, path)
        required = ("state_dict", "optimizer", "scheduler", "epoch", "best_metric", "wait", "stopped_early", "fingerprint", "history", "rng_state")
        first = load_checkpoint_checked(path, fingerprint, required)
        second = load_checkpoint_checked(path, fingerprint, required)
        if first["epoch"] != second["epoch"] or not tensors_finite(second["state_dict"]):
            raise AssertionError("checkpoint restore/idempotency check failed")
        corrupt = Path(temp_dir) / "corrupt.pt"
        corrupt.write_bytes(b"not a torch checkpoint")
        try:
            load_checkpoint_checked(corrupt, fingerprint, required)
            raise AssertionError("corrupt checkpoint unexpectedly loaded")
        except RuntimeError:
            pass
        rejected = [path for path in Path(temp_dir).glob("corrupt.pt.rejected_*") if path.suffix != ".json"]
        if len(rejected) != 1:
            raise AssertionError("corrupt checkpoint was not retained and isolated")
    return {"atomic_restore": True, "completed_resume_idempotent": True, "corrupt_checkpoint_isolated": True}


def synthetic_residual_checks() -> dict[str, Any]:
    rng = np.random.default_rng(42)
    groups = np.asarray([64, 64, 64], dtype=np.int64)
    target = rng.normal(size=int(groups.sum())).astype(np.float32)
    anchor = target + rng.normal(scale=1.0, size=target.size).astype(np.float32)
    good_residual = target - anchor
    anchor_ic = evaluate_rankic(anchor, target, groups)
    good_ic = evaluate_rankic(residual_correct(anchor, good_residual, groups, 0.5), target, groups)
    bad_ic = evaluate_rankic(residual_correct(anchor, -good_residual, groups, 0.5), target, groups)
    if not np.array_equal(residual_correct(anchor, good_residual, groups, 0.0), anchor):
        raise AssertionError("alpha=0 does not reproduce anchor")
    if good_ic <= anchor_ic or bad_ic >= anchor_ic:
        raise AssertionError("synthetic residual direction check failed")
    return {"anchor_rankic": anchor_ic, "good_rankic": good_ic, "reversed_rankic": bad_ic}


def full_control_flow_dry_run() -> dict[str, Any]:
    groups = np.asarray([5, 4], dtype=np.int64)
    rows = np.arange(int(groups.sum()), dtype=np.int64)
    anchor = np.linspace(0.05, 0.95, rows.size, dtype=np.float32)
    target = anchor[::-1].copy()
    residual = target - anchor
    calls = []
    fake_contract = CacheContract({}, {}, {
        "train": {"groups": groups, "rows": int(groups.sum()), "n_times": 2, "time_min": 486, "time_max": 487},
        "valid": {"groups": groups, "rows": int(groups.sum()), "n_times": 2, "time_min": 2918, "time_max": 2919},
        "test": {"groups": groups, "rows": int(groups.sum()), "n_times": 2, "time_min": 3161, "time_max": 3162},
    })
    fake_bundle = {"official_valid": {"blend_rank": anchor}, "test": {"rank": anchor}, "folds": {}}
    namespace = globals()
    original_load = np.load
    replacements = {
        "TRAIN_ENABLED": True,
        "validate_processed_cache": lambda: fake_contract,
        "load_anchor_or_rebuild": lambda *args, **kwargs: fake_bundle,
        "compute_normalization_stats": lambda *args, **kwargs: {"fingerprint": "dry-run"},
        "save_normalization_stats": lambda *args, **kwargs: None,
        "sha256_file": lambda *args, **kwargs: "unchanged",
        "write_csv": lambda *args, **kwargs: None,
        "save_sparse_grid": lambda *args, **kwargs: Path("dry-run.npy"),
        "validate_full_outputs": lambda: {"dry-run": {"bytes": 1, "sha256": "dry-run"}},
        "json_dump": lambda *args, **kwargs: None,
        "train_stage": lambda stage, seed, *args, **kwargs: (calls.append((stage, int(seed))) or object(), 7, 0.01),
        "predict_oof_interval": lambda *args, **kwargs: (anchor, residual, groups, rows),
        "oof_anchor_interval": lambda *args, **kwargs: (anchor, groups, rows),
        "predict_split_by_time": lambda *args, **kwargs: residual.copy(),
    }
    originals = {name: namespace[name] for name in replacements}
    try:
        namespace.update(replacements)
        np.load = lambda path, *args, **kwargs: target.copy() if Path(path).name in {"train_y.npy", "valid_y.npy"} else original_load(path, *args, **kwargs)
        report = run_full()
    finally:
        np.load = original_load
        namespace.update(originals)
    expected = [(stage, int(seed)) for stage in ("early", "dev", "shadow", "final") for seed in FULL_SEEDS]
    if calls != expected or "metrics" not in report or report["metrics"]["selected_alpha"] < 0:
        raise AssertionError("full control-flow dry run failed")
    return {"stage_calls": len(calls), "stages": ["early", "dev", "shadow", "final"], "selected_alpha_nonnegative": True}


def mini_checkpoint_check(contract: CacheContract, anchor_bundle: Mapping[str, Any]) -> dict[str, Any]:
    intervals = [("fold_1", 2189, 2192)]
    stats = compute_normalization_stats(contract, anchor_bundle, intervals)
    history = []
    model, epoch, metric = train_stage("preflight_mini", 42, contract, anchor_bundle, intervals, stats, fixed_epochs=2, history=history, resume=False)
    fingerprint = config_fingerprint("preflight_mini", 42, intervals, 2, stats)
    model_dir = RUN_DIR / "models_v4" / "preflight_mini" / "seed_42" / fingerprint[:16]
    path = model_dir / "best.pt"
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    restored = MaskedTemporalRankGLU().to(DEVICE); restored.load_state_dict(checkpoint["state_dict"])
    if not all(torch.equal(model.state_dict()[k].detach().cpu(), restored.state_dict()[k].detach().cpu()) for k in model.state_dict()):
        raise AssertionError("checkpoint restore mismatch")
    # Exercise the same anchor-aware inference path used by run_full().
    prediction_anchor, prediction_groups, _ = slice_oof_fold(anchor_bundle, "fold_1", contract, 2189, 2190)
    prediction = predict_split_by_time(restored, "train", contract, stats, prediction_anchor, 2189, 2190)
    if prediction.shape != prediction_anchor.shape or not np.isfinite(prediction).all():
        raise AssertionError("anchor-aware prediction regression check failed")
    resume_path = model_dir / "resume.pt"
    resume_checkpoint = torch.load(resume_path, map_location="cpu", weights_only=False)
    expected_fingerprint = config_fingerprint("preflight_mini", 42, intervals, 2, stats)
    if checkpoint.get("fingerprint") != expected_fingerprint or resume_checkpoint.get("fingerprint") != expected_fingerprint:
        raise AssertionError("checkpoint fingerprint mismatch")
    resumed_history: list[dict[str, Any]] = []
    resumed_model, resumed_epoch, _ = train_stage(
        "preflight_mini", 42, contract, anchor_bundle, intervals, stats,
        fixed_epochs=2, history=resumed_history, resume=True,
    )
    if resumed_epoch != epoch or len(resumed_history) != 2:
        raise AssertionError("completed checkpoint did not resume idempotently")
    del resumed_model
    return {
        "epochs": epoch, "metric": metric, "history_rows": len(history), "checkpoint": str(path),
        "prediction_rows": int(prediction.size), "prediction_groups": int(len(prediction_groups)),
        "prediction_finite": True, "checkpoint_fingerprint_valid": True,
        "completed_checkpoint_idempotent": True,
    }


def run_preflight() -> dict[str, Any]:
    before = sha256_file(FINAL_SUBMISSION)
    contract = validate_processed_cache()
    anchor_bundle = load_anchor_or_rebuild(contract, require_oof=True)
    folds = {}
    for name in OOF_INTERVALS:
        anchor, groups, rows = oof_anchor_interval(anchor_bundle, name, contract)
        folds[name] = {"rows": int(anchor.size), "groups": int(len(groups)), "row_start": int(rows[0]), "row_stop": int(rows[-1] + 1)}
    report = {
        "status": "preflight_passed", "trained": False, "folds": folds,
        "model": smoke_model_checks(), "synthetic": synthetic_residual_checks(),
        "full_control_flow": full_control_flow_dry_run(),
        "mini_training": mini_checkpoint_check(contract, anchor_bundle),
        "final_submission_sha256_before": before, "final_submission_sha256_after": sha256_file(FINAL_SUBMISSION),
    }
    if report["final_submission_sha256_before"] != report["final_submission_sha256_after"]:
        raise AssertionError("formal submission was modified")
    json_dump(RUN_DIR / "preflight_report.json", report)
    print(json.dumps({"status": report["status"], "trained": False}, ensure_ascii=False))
    return report


def run_smoke() -> dict[str, Any]:
    report = {
        "status": "smoke_passed", "trained": False, "real_data_read": False,
        "model": smoke_model_checks(), "synthetic": synthetic_residual_checks(),
        "stability": synthetic_stability_checks(), "checkpoint": synthetic_checkpoint_checks(),
        "full_control_flow": full_control_flow_dry_run(),
    }
    json_dump(RUN_DIR / "smoke_report.json", report)
    return report


def run_full() -> dict[str, Any]:
    if not TRAIN_ENABLED:
        raise RuntimeError("full mode is not enabled")
    before = sha256_file(FINAL_SUBMISSION)
    started = time.time()
    contract = validate_processed_cache()
    anchor_bundle = load_anchor_or_rebuild(contract, require_oof=True)
    train_y = np.load(CACHE_DIR / "common" / "train_y.npy", mmap_mode="r", allow_pickle=False)
    valid_y = np.asarray(np.load(CACHE_DIR / "common" / "valid_y.npy", mmap_mode="r", allow_pickle=False), dtype=np.float32)
    history: list[dict[str, Any]] = []

    early_train = [("fold_1", 2189, 2383)]
    early_valid = [("fold_1", 2383, 2432)]
    early_stats = compute_normalization_stats(contract, anchor_bundle, early_train)
    early_epochs = []
    for seed in FULL_SEEDS:
        model, best_epoch, _ = train_stage("early", int(seed), contract, anchor_bundle, early_train, early_stats, validation_intervals=early_valid, history=history)
        early_epochs.append(best_epoch); del model
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    fixed_epochs = int(np.median(early_epochs))

    fold1 = [("fold_1", 2189, 2432)]
    dev_stats = compute_normalization_stats(contract, anchor_bundle, fold1)
    dev_residuals = []
    for seed in FULL_SEEDS:
        model, _, _ = train_stage("dev", int(seed), contract, anchor_bundle, fold1, dev_stats, fixed_epochs=fixed_epochs, history=history)
        _, residual, dev_groups, dev_rows = predict_oof_interval(model, "fold_2", contract, anchor_bundle, dev_stats)
        dev_residuals.append(residual); del model
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    dev_anchor, dev_groups, dev_rows = oof_anchor_interval(anchor_bundle, "fold_2", contract)
    dev_residual = ensemble_standardized(dev_residuals, dev_groups)
    selected_alpha, weight_rows = select_alpha(dev_anchor, dev_residual, np.asarray(train_y[dev_rows], dtype=np.float32), dev_groups)
    write_csv(RUN_DIR / "weight_search.csv", weight_rows)
    dev_delta = next(row["delta"] for row in weight_rows if row["alpha"] == selected_alpha)

    fold12 = [("fold_1", 2189, 2432), ("fold_2", 2432, 2675)]
    shadow_stats = compute_normalization_stats(contract, anchor_bundle, fold12)
    shadow_residuals, shadow_seed_deltas = [], []
    shadow_anchor, shadow_groups, shadow_rows = oof_anchor_interval(anchor_bundle, "fold_3", contract)
    shadow_target = np.asarray(train_y[shadow_rows], dtype=np.float32)
    shadow_anchor_ic = evaluate_rankic(shadow_anchor, shadow_target, shadow_groups)
    for seed in FULL_SEEDS:
        model, _, _ = train_stage("shadow", int(seed), contract, anchor_bundle, fold12, shadow_stats, fixed_epochs=fixed_epochs, history=history)
        residual = predict_split_by_time(model, "train", contract, shadow_stats, shadow_anchor, 2675, 2918)
        shadow_residuals.append(residual)
        shadow_seed_deltas.append(evaluate_rankic(residual_correct(shadow_anchor, residual, shadow_groups, selected_alpha), shadow_target, shadow_groups) - shadow_anchor_ic)
        del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    shadow_residual = ensemble_standardized(shadow_residuals, shadow_groups)
    shadow_candidate = residual_correct(shadow_anchor, shadow_residual, shadow_groups, selected_alpha)
    shadow_delta = evaluate_rankic(shadow_candidate, shadow_target, shadow_groups) - shadow_anchor_ic

    final_intervals = [("fold_1", 2189, 2432), ("fold_2", 2432, 2675), ("fold_3", 2675, 2918)]
    final_stats = compute_normalization_stats(contract, anchor_bundle, final_intervals)
    save_normalization_stats(RUN_DIR / "normalization_stats.npz", final_stats)
    valid_anchor = anchor_bundle["official_valid"]["blend_rank"]
    test_anchor = anchor_bundle["test"]["rank"]
    valid_groups = contract.splits["valid"]["groups"]
    test_groups = contract.splits["test"]["groups"]
    valid_residuals, test_residuals = [], []
    for seed in FULL_SEEDS:
        model, _, _ = train_stage("final", int(seed), contract, anchor_bundle, final_intervals, final_stats, fixed_epochs=fixed_epochs, history=history)
        valid_residuals.append(predict_split_by_time(model, "valid", contract, final_stats, valid_anchor))
        test_residuals.append(predict_split_by_time(model, "test", contract, final_stats, test_anchor))
        del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    valid_residual = ensemble_standardized(valid_residuals, valid_groups)
    test_residual = ensemble_standardized(test_residuals, test_groups)
    valid_candidate = residual_correct(valid_anchor, valid_residual, valid_groups, selected_alpha)
    test_candidate = residual_correct(test_anchor, test_residual, test_groups, selected_alpha)
    valid_anchor_ic = evaluate_rankic(valid_anchor, valid_y, valid_groups)
    valid_candidate_ic = evaluate_rankic(valid_candidate, valid_y, valid_groups)
    valid_delta = valid_candidate_ic - valid_anchor_ic
    valid_anchor_diag = rankic_diagnostics(valid_anchor, valid_y, valid_groups)
    valid_candidate_diag = rankic_diagnostics(valid_candidate, valid_y, valid_groups)
    test_corr = mean_cross_sectional_rank_correlation(test_candidate, test_anchor, test_groups)
    sample_stride = max(test_residual.size // 250_000, 1)
    unique_ratio = float(np.unique(test_residual[::sample_stride]).size / max(test_residual[::sample_stride].size, 1))
    gates = {
        "alpha_positive": selected_alpha > 0,
        "dev_delta_min": dev_delta >= 0.0005,
        "shadow_nonnegative": shadow_delta >= 0,
        "official_valid_delta_min": valid_delta >= 0.0003,
        "official_valid_late": valid_candidate_diag["second_half"] >= valid_anchor_diag["second_half"] - 0.0005,
        "official_valid_worst_quarter": valid_candidate_diag["worst_quarter"] >= valid_anchor_diag["worst_quarter"] - 0.0015,
        "shadow_seed_std": float(np.std(shadow_seed_deltas)) <= 0.001,
        "test_anchor_correlation": test_corr >= 0.98,
        "residual_finite": bool(np.isfinite(valid_residual).all() and np.isfinite(test_residual).all()),
        "residual_not_collapsed": unique_ratio >= 0.05,
    }
    promoted = bool(all(gates.values()))
    prediction = test_candidate if promoted else test_anchor
    valid_prediction = valid_candidate if promoted else valid_anchor
    save_sparse_grid(test_anchor, "test", contract, "anchor_prediction.npy")
    save_sparse_grid(test_candidate, "test", contract, "candidate_prediction.npy")
    save_sparse_grid(prediction, "test", contract, "prediction.npy")
    save_sparse_grid(test_residual, "test", contract, "expert_residual.npy", fill_value=0.0)
    save_sparse_grid(valid_prediction, "valid", contract, "valid_prediction.npy")
    write_csv(RUN_DIR / "epoch_history.csv", history)
    metrics = {
        "fixed_epochs": fixed_epochs, "early_best_epochs": early_epochs, "selected_alpha": selected_alpha,
        "dev_delta": dev_delta, "shadow_delta": shadow_delta, "shadow_seed_deltas": shadow_seed_deltas,
        "valid_anchor_rankic": valid_anchor_ic, "valid_candidate_rankic": valid_candidate_ic, "valid_delta": valid_delta,
        "valid_anchor_diagnostics": valid_anchor_diag, "valid_candidate_diagnostics": valid_candidate_diag,
        "test_anchor_rank_correlation": test_corr, "residual_unique_ratio": unique_ratio, "gates": gates, "promoted": promoted,
    }
    choice = {
        "status": "PROMOTED: prediction.npy is v2 candidate" if promoted else "REJECTED: prediction.npy is anchor fallback",
        "promoted": promoted, "selected_alpha": selected_alpha,
        "candidate_file": str(RUN_DIR / "candidate_prediction.npy"), "recommended_file": str(RUN_DIR / "prediction.npy"),
        "fallback_used": not promoted,
    }
    json_dump(RUN_DIR / "metrics.json", metrics); json_dump(RUN_DIR / "submission_choice.json", choice)
    if before != sha256_file(FINAL_SUBMISSION):
        raise AssertionError("formal submission was modified")
    output_contract = validate_full_outputs()
    metadata = {
        "experiment": "exp_014_anchor_residual_rankglu_v2", "run_mode": RUN_MODE, "trained": True,
        "status": "full_completed", "max_epochs": MAX_EPOCHS, "fixed_epochs": fixed_epochs,
        "seeds": list(FULL_SEEDS), "selected_alpha": selected_alpha, "promoted": promoted,
        "fallback_used": not promoted, "primary_precision": PRIMARY_PRECISION,
        "checkpoint_schema_version": CHECKPOINT_SCHEMA_VERSION, "stability_policy": STABILITY_POLICY,
        "duration_seconds": round(time.time() - started, 3),
        "final_submission_sha256_before": before, "final_submission_sha256_after": sha256_file(FINAL_SUBMISSION),
        "outputs": output_contract,
    }
    json_dump(RUN_DIR / "metadata.json", metadata)
    print(json.dumps(choice, ensure_ascii=False))
    return {"metrics": metrics, "submission_choice": choice}

In [14]:
if RUN_MODE == "smoke":
    SMOKE_REPORT = run_smoke()
elif RUN_MODE == "preflight":
    PREFLIGHT_REPORT = run_preflight()
elif RUN_MODE == "full":
    FULL_REPORT = run_full()
else:
    raise ValueError(f"unknown RUN_MODE={RUN_MODE!r}")

processed cache contract: OK
{'train': {'rows': 6489099, 'n_times': 2432, 'time_min': 486, 'time_max': 2917}, 'valid': {'rows': 982972, 'n_times': 243, 'time_min': 2918, 'time_max': 3160}, 'test': {'rows': 2042538, 'n_times': 442, 'time_min': 3161, 'time_max': 3602}}
{'stage': 'early', 'seed': 42, 'epoch': 1, 'train_loss': 2.001743738798751, 'selection_metric': 0.14242833637731114, 'best_metric': 0.14242833637731114, 'lr': 0.000125, 'improved': True, 'primary_precision': 'bf16', 'fp32_retries': 0}
{'stage': 'early', 'seed': 42, 'epoch': 2, 'train_loss': 1.9926146066065915, 'selection_metric': 0.14280072651637896, 'best_metric': 0.14280072651637896, 'lr': 0.0005, 'improved': True, 'primary_precision': 'bf16', 'fp32_retries': 0}
{'stage': 'early', 'seed': 42, 'epoch': 3, 'train_loss': 1.9880516615110575, 'selection_metric': 0.14317121685014939, 'best_metric': 0.14317121685014939, 'lr': 0.0004979043378581745, 'improved': True, 'primary_precision': 'bf16', 'fp32_retries': 0}
{'stage': 'ear